In [1]:

import pandas as pd
import numpy as np


def recode_attributes(attribute):
    if attribute == 'Non-Failing Donor':
        return 'NF'
    elif attribute == 'Peripartum cardiomyopathy (PPCM)':
        return 'PPCM'
    elif attribute == 'Dilated cardiomyopathy (DCM)':
        return 'DCM'
    elif attribute == 'Hypertrophic cardiomyopathy (HCM)':
        return 'HCM'
    else:
        return 'W'



biomart_path = '/data/bionets/og86asub/alternet-project/alternet/data/biomart.txt'
biomart = pd.read_csv(biomart_path, sep='\t')

appris_path = '/data/bionets/mi34qaba/SpliceAwareGRN/data/appris_data.appris.txt'
appris_df = pd.read_csv(appris_path, sep='\t')

# sample attributions
sample_information_path = "/data/bionets/og86asub/alternet-project/alternet/data/run_table.csv"
sample_attributes = pd.read_csv(sample_information_path)
sample_attributes = sample_attributes.loc[:, ['Run', 'etiology']]
sample_attributes['eti'] = sample_attributes['etiology'].apply(lambda x: recode_attributes(x))

# TMPs rename columns
ktmp = pd.read_csv("/data/bionets/mi34qaba/data/MAGNet/MAGNet_transcript/MAGNet_kallisto_tpms", sep = '\t')
ktmp = ktmp.rename(columns={'target_id': 'transcript_id'})
names = [x.replace('_est_tpms', '') for x in ktmp.columns]
ktmp.columns = names

# Add gene ID
ktmp = ktmp.merge(biomart.loc[: ,['Transcript stable ID', 'Gene stable ID']], left_on ='transcript_id', right_on ='Transcript stable ID')
ktmp = ktmp.drop(columns=['Transcript stable ID'])
ktmp = ktmp.rename(columns={"Gene stable ID":'gene_id'})

# subset to protein coding transcripts
ktmp = ktmp[ktmp.transcript_id.isin(appris_df[appris_df['Transcript type']=='protein_coding']['Transcript ID'])]


In [26]:

conditions = ['DCM', 'HCM', 'NF']

for condi in conditions:

    # subset data to condition
    samples = sample_attributes[sample_attributes['eti'] == condi]
    samples = samples['Run'].tolist()

    transcript_data_cp = ktmp.loc[:, ['transcript_id', 'gene_id'] + samples].copy(deep=True)

    nonzero_count = np.count_nonzero(transcript_data_cp.values, axis=1)
    frac = nonzero_count/transcript_data_cp.shape[1]
    transcript_data_cp = transcript_data_cp.iloc[np.where(frac>0.50)[0]]

    transcript_data_cp = transcript_data_cp.reset_index()
    transcript_data_cp =transcript_data_cp.drop(columns=['index'])
    print(transcript_data_cp.shape)
    print(len(transcript_data_cp.gene_id.unique()))
    print(len(transcript_data_cp.transcript_id.unique()))

    print(len(transcript_data_cp[transcript_data_cp.transcript_id.isin(tf_list['Transcript stable ID'])]['gene_id'].unique()))
    print(len(transcript_data_cp[transcript_data_cp.transcript_id.isin(tf_list['Transcript stable ID'])]['transcript_id'].unique()))

    transcript_data_cp.to_csv(f"/data/bionets/og86asub/alternet-project/alternet/data/{condi}_magnet_prefiltered_tpm.tsv", sep = '\t', index = False)


(40999, 168)
15955
40999
1539
4191
(42783, 30)
16111
42783
1549
4411
(41989, 168)
15827
41989
1530
4332


In [15]:
transcript_data_cp

,SRR10676821,SRR10676822,SRR10676823,SRR10676824,SRR10676825,SRR10676826,SRR10676827,SRR10676828,SRR10676829,SRR10676830,...,SRR10677161,SRR10677163,SRR10677168,SRR10677171,SRR10677174,SRR10677175,SRR10677176,SRR10677177,SRR10677179,SRR10677184
0,18.182400,23.665400,58.730700,10.415700,43.194400,55.551300,44.604100,29.813400,32.030600,133.887000,...,7.469420,53.473300,87.091700,7.828650,142.587000,86.153100,15.702000,33.564000,149.306000,9.579430
1,0.292345,0.181541,0.883829,0.243651,0.604892,0.508725,0.322050,0.000000,0.211454,0.741523,...,0.663738,0.207567,0.544754,0.424345,1.000220,0.116289,0.400908,0.554897,0.418136,0.433982
2,0.000000,0.000000,0.359728,0.000000,0.649254,0.000000,0.210922,0.000000,0.000000,0.194211,...,0.000000,0.612003,0.000000,0.000000,1.275920,1.083740,0.161790,0.000000,1.907470,0.000000
3,1.374890,0.889535,0.034239,0.571389,0.294203,0.925668,0.805371,2.142800,2.034280,0.884599,...,0.526265,0.390488,1.620090,1.165040,1.114420,0.749224,1.136220,2.119570,0.329100,0.828792
4,0.583894,0.140139,0.000000,0.225796,0.023858,0.011475,0.082714,0.591058,0.106950,0.067095,...,0.098087,0.215930,0.269296,0.264715,0.016233,0.069791,0.477872,0.198170,0.018419,0.252184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41744,0.818590,0.585078,0.654106,0.263536,1.351650,1.123820,1.441250,0.736727,0.853860,4.621250,...,0.646817,0.985754,0.461176,0.295557,1.523890,0.264451,0.485606,0.272582,3.784560,0.452603
41745,0.050290,0.014330,0.035171,0.041823,0.037060,0.000000,0.065104,0.016308,0.026477,0.072751,...,0.051768,0.040899,0.000000,0.036091,0.000000,0.022102,0.044203,0.044029,0.000000,0.051400
41746,0.397195,0.139399,0.137387,0.238294,0.456423,0.055239,0.127514,0.330749,0.091594,0.087561,...,0.067772,0.278306,0.285876,0.041657,0.033892,0.187877,0.468567,1.001040,0.049527,0.233459
41747,0.769121,0.759989,0.346327,0.559473,0.161998,0.323150,1.094700,0.756638,0.679176,0.024063,...,0.592340,0.360954,0.000000,0.103718,0.098629,0.434144,0.721068,0.476099,0.116454,0.425519


In [19]:

from alternet.annotation import map_tf_ids
tf_list_path = '/data/bionets/og86asub/alternet-project/alternet/data/allTFs_hg38.txt'

tf_list = pd.read_csv(tf_list_path, sep='\t', header = None)
tf_list = map_tf_ids(tf_list, biomart)




In [25]:
len(transcript_data_cp[transcript_data_cp.transcript_id.isin(tf_list['Transcript stable ID'])]['gene_id'].unique())

1530